# 06 — Relatório Final

**Objetivo:** Validar a integridade ponta a ponta (transação crua → classificação), consolidar todos os artefatos e gerar o relatório HTML final, deixando o pipeline pronto para deploy.

---

**Roteiro:**

1. Setup
2. Carregamento de dados
3. Teste de integridade ponta a ponta
4. Relatório final


### Etapa 1 - Setup

In [1]:
%load_ext autoreload
%autoreload 2

import json

import joblib
import numpy as np
import pandas as pd

from src.config import CONFIG, CAMINHOS
from src.report import RelatorioHTML
from src.viz_config import aplicar_tema_seaborn, PALETTE, CORES

DADOS_PRO   = CAMINHOS.dados_processed
MODELS_DIR  = CAMINHOS.modelos
FIGURAS_DIR = CAMINHOS.figures
REPORTS_DIR = CAMINHOS.reports

aplicar_tema_seaborn()

print("\n✅ Setup configurado!\n")

print(f"FIGURAS           : {FIGURAS_DIR}")
print(f"MODELOS           : {MODELS_DIR}")
print(f"RELATÓRIOS        : {REPORTS_DIR}")
print(f"DADOS PROCESSADOS : {DADOS_PRO}")


✅ Setup configurado!

FIGURAS           : C:\Users\jas_t\Repo\Portfolios\credit_card_fraud\reports\figures
MODELOS           : C:\Users\jas_t\Repo\Portfolios\credit_card_fraud\models
RELATÓRIOS        : C:\Users\jas_t\Repo\Portfolios\credit_card_fraud\reports
DADOS PROCESSADOS : C:\Users\jas_t\Repo\Portfolios\credit_card_fraud\data\processed


### Etapa 2 - Carregamento de dados

In [2]:
pipeline_final = joblib.load(MODELS_DIR / "pipeline_final.joblib")
print(f"Pipeline: {pipeline_final.named_steps['clf'].__class__.__name__}")

Pipeline: XGBClassifier


In [3]:
X_test = pd.read_parquet(DADOS_PRO / "X_test.parquet")
y_test = pd.read_parquet(DADOS_PRO / "y_test.parquet").iloc[:, 0]

print(f"Teste: {X_test.shape} · {y_test.sum()} fraudes")

Teste: (56746, 30) · 95 fraudes


In [4]:
meta   = json.load(open(MODELS_DIR / "metadados.json", encoding="utf-8"))

print(f"Threshold : {meta['threshold']:.3f}")
print(f"PR-AUC teste: {meta['pr_auc_teste']:.4f}")

Threshold : 0.264
PR-AUC teste: 0.8261


### Etapa 3 - Teste de integridade ponta a ponta

Como boa prática, podemos fazer  Teste de integridade ponta a ponta nos artefatos carregados

In [5]:
THRESHOLD = meta["threshold"]

def classificar_transacao(transacao_df):
    """Recebe transação(ões) crua(s), retorna predição + probabilidade.
    Replica exatamente o que o app fará: pipeline aplica preprocessor
    internamente, e o threshold do metadados decide fraude/legítima."""
    proba = pipeline_final.predict_proba(transacao_df)[:, 1]
    pred  = (proba >= THRESHOLD).astype(int)
    return pred, proba

print("═" * 60)
print("TESTE DE INTEGRIDADE PONTA A PONTA")
print("═" * 60)

# ── Caso 1: uma fraude real do teste ─────────────────────────────
idx_fraude = y_test[y_test == 1].index[0]
fraude_ex  = X_test.loc[[idx_fraude]]
pred_f, proba_f = classificar_transacao(fraude_ex)
print(f"\n   Transação FRAUDULENTA (real):")
print(f"     P(fraude) = {proba_f[0]:.3f} → "
      f"{'FRAUDE' if pred_f[0]==1 else 'legítima'} "
      f"{'✅' if pred_f[0]==1 else '⚠️'}")

# ── Caso 2: uma transação legítima do teste ─────────────────────
idx_legit  = y_test[y_test == 0].index[0]
legit_ex   = X_test.loc[[idx_legit]]
pred_l, proba_l = classificar_transacao(legit_ex)
print(f"\n   Transação LEGÍTIMA (real):")
print(f"     P(fraude) = {proba_l[0]:.3f} → "
      f"{'fraude' if pred_l[0]==1 else 'LEGÍTIMA'} "
      f"{'✅' if pred_l[0]==0 else '⚠️'}")


print("\n" + "═" * 60)
print("ARTEFATOS DE DEPLOY")
print("═" * 60)
artefatos = [
    MODELS_DIR / "pipeline_final.joblib",
    MODELS_DIR / "metadados.json",
]
for art in artefatos:
    existe = art.exists()
    tam = f"{art.stat().st_size/1024:.0f} KB" if existe else "—"
    print(f"   {'✅' if existe else '❌'} {art.name:<28} {tam:>10}")

════════════════════════════════════════════════════════════
TESTE DE INTEGRIDADE PONTA A PONTA
════════════════════════════════════════════════════════════

   Transação FRAUDULENTA (real):
     P(fraude) = 0.999 → FRAUDE ✅

   Transação LEGÍTIMA (real):
     P(fraude) = 0.000 → LEGÍTIMA ✅

════════════════════════════════════════════════════════════
ARTEFATOS DE DEPLOY
════════════════════════════════════════════════════════════
   ✅ pipeline_final.joblib           1056 KB
   ✅ metadados.json                     1 KB


### Etapa 4 - Relatório final

O relatorio consiste em um resumo pontual dos resultados obtidos com:
Tabela de métricas finais
Figuras curadas
aproveitando o arquivo RelatorioHTML.py

In [6]:
metricas_tab = pd.DataFrame({
    "Métrica": ["PR-AUC (CV)", "PR-AUC (teste)", "ROC-AUC (teste)",
                "Recall", "Precision", "F1", "Threshold"],
    "Valor": [f"{meta['pr_auc_cv']:.4f}", f"{meta['pr_auc_teste']:.4f}",
              f"{meta['roc_auc_teste']:.4f}", f"{meta['recall_teste']:.3f}",
              f"{meta['precision_teste']:.3f}", f"{meta['f1_teste']:.3f}",
              f"{meta['threshold']:.3f}"],
}).set_index("Métrica")

In [7]:
FIGS = [
    (FIGURAS_DIR / "nb01_desbalanceamento.png",      "Desbalanceamento extremo — 578:1 (0.17% fraude)"),
    (FIGURAS_DIR / "nb01_poder_discriminativo.png",  "Poder discriminativo (Mann-Whitney) — V14, V4, V12 lideram"),
    (FIGURAS_DIR / "nb03_comparativo_modelos.png",   "Comparação de modelos — XGBoost e RandomForest no topo (PR-AUC)"),
    (FIGURAS_DIR / "nb04_optuna.png",                "Tuning Optuna — XGBoost vence (PR-AUC CV 0.857)"),
    (FIGURAS_DIR / "nb04_avaliacao.png",             "Avaliação no teste — matriz de confusão + curvas PR/ROC"),
    (FIGURAS_DIR / "nb05_shap_summary.png",          "SHAP — V14/V4/V12 lideram (confirma a EDA)"),
    (FIGURAS_DIR / "nb05_waterfall_tp.png",          "Fraude detectada (P=1.0) — assinatura extrema"),
    (FIGURAS_DIR / "nb05_waterfall_fn.png",          "Fraude que escapou (P=0.0) — camuflada como legítima"),
]

In [8]:
caminho = (
    RelatorioHTML(
        titulo="Detecção de Fraude em Cartão de Crédito",
        autor="Jhonnes Toledo",
        subtitulo="Classificação desbalanceada · Pipeline sklearn + XGBoost · PR-AUC 0.83",
    )
    .add_secao("1. Visão geral",
        "<div class='card'><p>Detecção de fraude em <strong>284.807 transações</strong> "
        "de cartão (dataset Online Retail II / ULB) com desbalanceamento <strong>extremo "
        "de 578:1</strong> (0.17% de fraude). Um <code>Pipeline</code> sklearn "
        "(<code>RobustScaler → XGBoost</code>) classifica cada transação, com threshold "
        "otimizado para o trade-off de negócio recall/precision.</p></div>")
    .add_secao("2. Método",
        "<ul>"
        "<li><strong>EDA estatística:</strong> 30/30 features não-normais (D'Agostino) → "
        "Mann-Whitney; V14, V4, V12 mais discriminativas (efeito ~0.9).</li>"
        "<li><strong>Pré-processamento:</strong> RobustScaler no Amount (outliers até £25k), "
        "V1-V28 passthrough (já PCA), Time descartado.</li>"
        "<li><strong>Balanceamento:</strong> testadas 4 estratégias; PR-AUC idêntico "
        "(classes separáveis) → class_weight='balanced' (sem inflar dados).</li>"
        "<li><strong>Modelos:</strong> 7 comparados por PR-AUC; XGBoost e RandomForest no topo.</li>"
        "<li><strong>Tuning:</strong> Optuna (40 trials) — XGBoost vence (PR-AUC CV 0.857).</li>"
        "<li><strong>Threshold:</strong> otimizado para máximo F1 (0.264).</li>"
        "<li><strong>Interpretabilidade:</strong> SHAP confirma a EDA (V14>V4>V12).</li>"
        "</ul>")
    .add_secao("3. Métricas finais (teste intocado)", "")
    .add_tabela(metricas_tab, incluir_indice=True)
    .add_secao("4. Resultado em números",
        "<div class='card'><p>No teste (56.746 transações, 95 fraudes reais), com "
        "threshold 0.264: o modelo <strong>detecta 77 das 95 fraudes (81%)</strong> "
        "gerando apenas <strong>4 falsos alarmes</strong>. O ROC-AUC (0.974) parece alto, "
        "mas é o <strong>PR-AUC (0.826)</strong> que mede o desempenho real na classe rara.</p></div>")
    .add_secao("5. Insight: o limite do recall",
        "<div class='card'><p>O modelo acerta com certeza as fraudes de <strong>assinatura "
        "extrema</strong> (V14≈-13), mas tem um teto de recall ~80%: as fraudes que escapam "
        "são <strong>camufladas</strong> — seus componentes PCA (V14≈0) imitam transações "
        "legítimas, sem sinal detectável. Os 20% perdidos são um limite estrutural do "
        "problema, não falha de ajuste.</p></div>")
    .add_secao("6. Visualizações", "")
    .add_figuras(FIGS)
    .add_secao("7. Conclusão",
        "<div class='card'><p>Pipeline de classificação desbalanceada com validação "
        "estatística rigorosa (EDA → modelo → SHAP convergindo nas mesmas features) e "
        "decisão de threshold ancorada no negócio. O artefato "
        "<code>pipeline_final.joblib</code> recebe a transação crua e retorna a "
        "classificação em uma chamada, pronto para deploy.</p></div>")
    .add_html("<footer>Credit Card Fraud Detection · Pipeline sklearn + XGBoost · "
              "github.com/jhastoledo</footer>")
    .salvar(REPORTS_DIR / "relatorio_final.html")
)

In [9]:
print("═" * 60)
print("RELATÓRIO FINAL GERADO")
print("═" * 60)

print(f"   Salvo: reports/relatorio_final.html")
print(f"   Tamanho: {caminho.stat().st_size/1e6:.2f} MB · {len(FIGS)} figuras")

════════════════════════════════════════════════════════════
RELATÓRIO FINAL GERADO
════════════════════════════════════════════════════════════
   Salvo: reports/relatorio_final.html
   Tamanho: 0.70 MB · 8 figuras


### Etapa 5 - Fechamento

**ARTEFATOS DE DEPLOY**
* models/pipeline_final.joblib (RobustScaler → XGBoost afinado)
* models/metadados.json (threshold 0.264 + métricas)
* reports/relatorio_final.html (0.70 MB, 8 figuras)

**TESTE DE INTEGRIDADE**
* Fraude real    → P=0.999 → FRAUDE
* Legítima real  → P=0.000 → LEGÍTIMA
* Inferência ponta a ponta validada (transação crua → classe)

**REUTILIZAÇÃO COMPROVADA**
* src/report.py reusado SEM alteração (3º projeto: 2 cluster + 1 classif)
* viz_config com aplicar_tema_seaborn (matplotlib + seaborn integrados)

**STATUS: ✅ 6 NOTEBOOKS COMPLETOS — PROJETO PRONTO PARA DEPLOY**

**RESULTADO FINAL**
* XGBoost · PR-AUC 0.826 (teste) · threshold 0.264
* 81% das fraudes detectadas · só 4 falsos alarmes em 56k